# Fine-tuning de modelo para sumarização de diálogos de atendimento ao cliente pelo Twitter

- Cria aplicação com Gradio e modelo "tunado" para sumarização de diálogos

- Testa aplicação no localhost

## Configuração inicial

In [1]:
MODEL_NAME = 'falconsai'
MODEL_PATH = r'/home/msc/Downloads/hf_models'

LOCAL_FT_MODEL_PATH = f'../models/{MODEL_NAME}_ft'
LOCAL_MODEL_PATH = f'{MODEL_PATH}/{MODEL_NAME}'

# %pip install gradio

In [ ]:
import torch
import gradio as gr
from rich import print
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import pandas as pd

## Exemplos

In [3]:
df_eval_ft = pd.read_csv('../data/processed/df_eval_ft_falconsai_30.csv')
print(df_eval_ft.shape)
df_eval_ft.head(1)

(866, 7)

,conversation_id,tweet_ids,created_at_list,tweet_texts,human_summary,elapsed_time,summary_pred
0,b065262210783596c1fe79466b8f8985,"[87068, 87069, 87072, 87076]","{87068: Timestamp('2017-11-30 14:29:00+0000', ...",So neither my iPhone nor my Apple Watch are re...,Health and activity functions are not working ...,1 days 17:43:08,The customer is complaining that they are not ...


In [8]:
print(df_eval_ft.iloc[1].to_dict())

{
    'conversation_id': '1e1d8fd4f95c984fb78687c9e946dc97',
    'tweet_ids': '[607293, 607296, 607297, 607297]',
    'created_at_list': "{607293: Timestamp('2017-11-22 12:16:27+0000', tz='UTC'), 607296: Timestamp('2017-11-22 
11:29:30+0000', tz='UTC'), 607297: Timestamp('2017-11-22 11:19:41+0000', tz='UTC')}",
    'tweet_texts': "@115850 hi team! i m planning to get Apple AirPods ! it shows on the website it has 10 days 
replacement warranty, can u explain me what is it ? @264322 We've a 10days replacement policy if the item you 
received is damaged or defective. ^SH @264322 Yes, headsets/ earphones are not eligible for remorse returns. In 
case of any damage/ defect you can reach out to us, we'll check and help you accordingly. ^VN",
    'human_summary': 'Customer is eager to know about the replacement policy on the earphones he wishes to buy. 
Agent stated that it only applies if the received item is defective or damaged.',
    'elapsed_time': '0 days 00:56:46',
    'summary_pred': 'the customer is complaining that he is unable to get the product on the website which has 10 
days replacement warranty. The agent asked the customer to explain what is the issue and said that the replacement 
policy is if the item received is damaged or defective.'
}

In [9]:
print(df_eval_ft.tweet_texts.iloc[0:3].to_dict())

{
    0: 'So neither my iPhone nor my Apple Watch are recording my steps/activity, and Health doesn’t recognise 
either source anymore for some reason. Any ideas? https://t.co/m9DPQbkftD @135060 Thank you. Have you tried 
restarting both devices since this started happening? @AppleSupport Yes, everything seems fine, it’s just Health 
and activity. @135060 Let’s move to DM and look into this a bit more. When reaching out in DM, let us know when 
this first started happening please. For example, did it start after an update or after installing a certain app? 
https://t.co/GDrqU22YpT',
    1: "@115850 hi team! i m planning to get Apple AirPods ! it shows on the website it has 10 days replacement 
warranty, can u explain me what is it ? @264322 We've a 10days replacement policy if the item you received is 
damaged or defective. ^SH @264322 Yes, headsets/ earphones are not eligible for remorse returns. In case of any 
damage/ defect you can reach out to us, we'll check and help you accordingly. ^VN",
    2: "@AskAmex Where do I write to address a customer service issue to higher management? @216929 Hi Chris. Which
U.S. based card is this concerning? Please do not release any personal or card information. ^Clarissa @AskAmex 
Signed up for new card with Delta to book immediately book tix. Card number didn't come up. Customer svce refused 
to help. @216929 Good morning, thanks for reaching out. Please call our New Accounts Team at 877-399-3086, for 
assistance. They're available,"
}

## Gradio App

In [6]:
# exemplo de input
print('''@XboxSupport For some odd reason. The vibration on my controller doesn't work when i play Destiny 2. I went through every setting but nothing
@364434 Would you mind sending us a picture of what you have the settings set to exactly?
@364434 Okay, that is odd. Was the controller vibrating with the game before or has it never vibrated?
@364434 the console: https://t.co/QENEhDawC3 and check again if the vibration is still disabled.''')

@XboxSupport For some odd reason. The vibration on my controller doesn't work when i play Destiny 2. I went through
every setting but nothing
@364434 Would you mind sending us a picture of what you have the settings set to exactly?
@364434 Okay, that is odd. Was the controller vibrating with the game before or has it never vibrated?
@364434 the console: https://t.co/QENEhDawC3 and check again if the vibration is still disabled.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(LOCAL_FT_MODEL_PATH)
model_ft = AutoModelForSeq2SeqLM.from_pretrained(
    LOCAL_FT_MODEL_PATH,
    dtype=torch.float32,
    device_map='auto'
)

def summarize_ft(input_text):
    if not input_text.strip():
        return 'Digite um texto.'

    prompt = f'summarize: {input_text}'
    
    # Tokenização (entrada)
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True
    ).to(model_ft.device)

    summary_ids = model_ft.generate(
        inputs['input_ids'], 
        max_new_tokens=100,
        do_sample=False,
        max_length=700, 
        min_length=40, 
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

    # Decodificação (saída)
    res = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return res


with gr.Blocks() as demo:
    gr.Markdown(
        "<h2 style='text-align: center;'>📋 Sumarizador de diálogos de clientes no Twitter</h2>"
    )

    with gr.Row():
        # Coluna esquerda
        with gr.Column():
            input_text = gr.Textbox(
                label='✍🏽 Texto de entrada',
                lines=12,
                placeholder='Texto com diálogo de atendimento ao cliente pelo Twitter...'
            )
            btn = gr.Button('Gerar sumarização')

        # Coluna direita
        with gr.Column():
            saida = gr.Textbox(
                label='🤖 Sumarização gerada',
                lines=12
            )

    btn.click(fn=summarize_ft, inputs=input_text, outputs=saida)

demo.launch()

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


**Resultado esperado**

<img src="../imgs/app_print.png">